# リークしないパイプライン

**出典**　機械学習 4 分野の教科書 — 第 4 章 1-C アンサンブル・評価・前処理 ／ 07 実装：リークしないパイプラインを組む

**このノートのパス**　`ml-code/04_ensemble_eval/03_leakproof_pipeline.ipynb`

**このノートで確かめること**
数値列とカテゴリ列で別々の前処理を定義し、**前処理とモデルを 1 つの Pipeline にまとめる**。
こうすると交差検証の各 fold の内側で fit されるため、**リークが構造的に起きない**。

| 項目 | 内容 |
|---|---|
| 必要なパッケージ | `scikit-learn, pandas, numpy` |
| 実行時間の目安 | 約 1 分 |
| 前提 | **このノート単体で完結する**（他のノートを先に実行する必要はない） |

In [2]:
# 未導入なら実行する（Colab など）
# !pip install -q scikit-learn pandas numpy

In [3]:
import numpy as np, pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (StratifiedKFold, GridSearchCV,
                                     cross_val_score, train_test_split)
from sklearn.metrics import roc_auc_score

# --- 動かして確かめられるよう、小さな擬似データを作る ---
rng = np.random.default_rng(0)
n = 600
X = pd.DataFrame({
    "age":           rng.integers(20, 70, n).astype(float),
    "income":        rng.normal(500, 150, n),
    "tenure_months": rng.integers(1, 120, n).astype(float),
    "prefecture":    rng.choice(["東京", "大阪", "福岡"], n),
    "plan_type":     rng.choice(["A", "B", "C"], n),
})
X.loc[rng.choice(n, 40, replace=False), "income"] = np.nan   # 欠損もわざと混ぜる
y = ((X["tenure_months"] < 48) & (X["plan_type"] != "C")).astype(int).to_numpy(copy=True)
# ルールどおり 100% 決まるデータだと AUC が 1.000 になって学習の様子が見えないので、
# 現実に合わせて 5% だけラベルを反転させる
flip = rng.choice(n, int(n * 0.05), replace=False)
y[flip] = 1 - y[flip]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=0)

In [4]:
num_cols = ["age", "income", "tenure_months"]
cat_cols = ["prefecture", "plan_type"]

# 数値列とカテゴリ列で別々の前処理を定義する
numeric = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale",  StandardScaler()),
])
categorical = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),   # 本番で未知の値が来ても落ちない
])

pre = ColumnTransformer([
    ("num", numeric,     num_cols),
    ("cat", categorical, cat_cols),
])

# ★ 前処理とモデルを 1 つの Pipeline にまとめるのが肝。
#   こうすると交差検証の各 fold の内側で fit されるため、リークが構造的に起きない。
pipe = Pipeline([
    ("pre",   pre),
    ("model", RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1)),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

# 1) まず素の性能を測る
scores = cross_val_score(pipe, X_tr, y_tr, cv=cv, scoring="roc_auc")
print(f"ベースライン AUC: {scores.mean():.3f} ± {scores.std():.3f}")

ベースライン AUC: 0.939 ± 0.040


In [5]:
# 2) ハイパーパラメータ探索も Pipeline ごと行う
grid = GridSearchCV(
    pipe,
    param_grid={
        "model__max_depth":        [None, 8, 16],
        "model__min_samples_leaf": [1, 5, 20],
    },
    cv=cv, scoring="roc_auc", n_jobs=-1,
)
grid.fit(X_tr, y_tr)
print("最良パラメータ:", grid.best_params_)
print("交差検証 AUC :", grid.best_score_)

# 3) テストデータに触るのは、ここで一度だけ
print("テスト AUC   :", roc_auc_score(y_te, grid.predict_proba(X_te)[:, 1]))

最良パラメータ: {'model__max_depth': None, 'model__min_samples_leaf': 1}
交差検証 AUC : 0.939177258950805
テスト AUC   : 0.964506675350049


## 読み取りどころ

- **`<推定器名>__<パラメータ名>` という二重アンダースコア記法**が、Pipeline の中の部品を指す。
  `model__max_depth` は「`model` という名前のステップの `max_depth`」。
- **`handle_unknown="ignore"`** は本番で効く。学習時に見なかったカテゴリが来ても落ちず、全 0 のベクトルになる。これが無いと本番で例外が飛ぶ。
- **欠損補完も Pipeline の中に入れる。**外で `fillna` すると、全データの中央値を使ってしまい検証データの情報が漏れる。

この Pipeline の組み方は **1-D の探索でもそのまま使う**（`01D_hpo/04_sklearn_search.ipynb`）。